# EnergyPrime: быстрый старт

Весь путь из ноутбуков SVGP / LSTM / XGBoost, но через библиотеку:

1. данные + спецификация станции;
2. качество данных (NaN, нули, длины серий);
3. очистка (пробный прогон → split → статистики только по train);
4. модель по рецепту, прогноз, метрики по дням;
5. сохранение / загрузка (так будет работать GUI);
6. LSTM и SVGP, лаги мощности и рекурсивный прогноз, сравнение моделей.

Перед запуском: `pip install -e "..[all]"` в вашем окружении (один раз).

In [ ]:
from pathlib import Path

import pandas as pd

from energyprime import Cleaner, Forecaster, load_station, split_by_time
from energyprime import analysis, evaluation
from energyprime import visualization as viz

DATA_DIR = Path("../../../Orig_data")   # папка с solar_dataset_full_*.csv — поменяйте на свою
SPECS = Path("specs")
RUNS = Path("../runs")                   # сюда сохраняются модели (папка в .gitignore)

## 1. Данные и спецификация станции

Спецификация (`specs/china_site1_hourly.yaml`) говорит, что target, какие колонки — признаки,
их единицы и физические пределы, установленная мощность и правила очистки.

In [ ]:
ds = load_station(DATA_DIR / "solar_dataset_full_1.csv", SPECS / "china_site1_hourly.yaml")
print(ds)
ds.info

## 2. Качество данных

In [ ]:
analysis.describe_columns(ds, ds.spec.used_columns)

In [ ]:
# сколько серий нулей/NaN какой длины — по этой таблице выбирают пороги правил
analysis.run_length_table(ds, ["POWER", "GHI", "RH"], nan=True, zeros=True, merge=True).head(15)

In [ ]:
viz.plot_missing_share(ds, ds.spec.used_columns, zeros=True)

In [ ]:
viz.plot_time_series(ds, ["POWER", "GHI", "AIR_T", "RH"], missing="nan")

## 3. Очистка

Сначала пробный прогон: что сделают правила, данные не меняются. Потом разбиение по времени
и `fit` только на train — статистики выбросов (z-score) не видят тестовый период.

In [ ]:
cleaner = Cleaner.from_spec(ds.spec)
plan = cleaner.plan(ds)
print("строк будет удалено:", plan["rows_to_drop"])
plan["runs"]

In [ ]:
split = split_by_time(ds, test="30D", val="5D")
cleaner.fit(split.train)
split = cleaner.transform(split)
reports = cleaner.get_report()          # отчёт по каждой части: train / val / test
print(split)
reports["train"]["fills"]

In [ ]:
reports["train"]["outliers"]

In [ ]:
viz.plot_mask_heatmap(reports["train"]["outlier_mask"], title="Выбросы, заменённые интерполяцией")

## 4. Модель по рецепту

Рецепт = модель + подходящая ей подготовка данных. `xgboost`: погода + YEAR + циклические час
и день года, без масштабирования. Любой параметр можно переопределить.

In [ ]:
fc = Forecaster.from_recipe("xgboost")
fc.fit(split.train, validation=split.val)
print(fc)
fc.info_["features"]

In [ ]:
viz.plot_learning_curve(fc.history_, title="XGBoost")

## 5. Прогноз и метрики

`history` — данные перед тестовым периодом: нужны лагам и окнам LSTM в начале прогноза.
Нормировка — на установленную мощность из спецификации (если её нет — на максимум).

In [ ]:
history = pd.concat([split.train.data, split.val.data])
pred = fc.predict(split.test, history=history)
fc.evaluate(split.test, history=history, percent=True)

In [ ]:
daily = evaluation.evaluate_by_window(split.test.y, pred, window=24, norm=ds.capacity, percent=True)
evaluation.summarize_windows(daily)

In [ ]:
viz.plot_forecast(split.test.y[-120:], pred[-120:], title="XGBoost: последние 5 дней")

In [ ]:
viz.plot_window_metrics(daily, "nrmse", title="nRMSE по суткам, %")

In [ ]:
viz.plot_feature_importance(fc.feature_importance())

## 6. Сохранение и загрузка — так прогноз будет получать GUI

In [ ]:
fc.save(RUNS / "xgb_site1")
loaded = Forecaster.load(RUNS / "xgb_site1")
loaded.required_inputs

## 7. LSTM и SVGP (обучаются дольше)

Обе модели хранят веса лучшей и последней эпохи: `predict(..., weights="last")`.

In [ ]:
lstm = Forecaster.from_recipe("lstm", model_params={"epochs": 100, "seed": 42})
lstm.fit(split.train, validation=split.val)
lstm.evaluate(split.test, history=history, percent=True)

In [ ]:
svgp = Forecaster.from_recipe("svgp", model_params={"epochs": 200, "seed": 42})
svgp.fit(split.train, validation=split.val)
interval = svgp.predict_interval(split.test)          # pred, lower, upper (mean ± 1σ), std
last = interval.iloc[-120:]
viz.plot_forecast(split.test.y[-120:], last["pred"], last["lower"], last["upper"], title="SVGP")

In [ ]:
# CRPS как в ноутбуке SVGP (по гауссиане с std = (upper - lower) / 2)
evaluation.evaluate_regression(split.test.y, interval["pred"], ["nmae", "nrmse", "r2", "ncrps"],
                               norm=ds.capacity, std=interval["std"], percent=True)

## 8. История мощности как признак и рекурсивный прогноз

С лагами POWER обычный `predict` берёт лаги из настоящих значений (прогноз на 1 шаг).
`predict_recursive` подставляет свои же прогнозы — честный прогноз на весь горизонт.

In [ ]:
lagged = Forecaster.from_recipe("xgboost", lags={"POWER": [1, 24, 48, 120]},
                                rolling={"POWER": {"windows": [24], "stats": ["mean", "std"]}})
lagged.fit(split.train, validation=split.val)
five_days = split.test.data.iloc[:120]
one_step = lagged.predict(five_days, history=history)
recursive = lagged.predict_recursive(five_days, history=history)
pd.DataFrame({
    "one_step": evaluation.evaluate_regression(five_days["POWER"], one_step, ["nmae", "nrmse", "r2"], norm=50, percent=True),
    "recursive": evaluation.evaluate_regression(five_days["POWER"], recursive, ["nmae", "nrmse", "r2"], norm=50, percent=True),
})

## 9. Сравнение моделей (будущий Benchmark) и проверка на утечку

In [ ]:
scores = {}
for name in ["linear", "random_forest", "xgboost"]:
    model = Forecaster.from_recipe(name).fit(split.train, validation=split.val)
    scores[name] = model.evaluate(split.test, history=history, percent=True)
pd.DataFrame(scores).T

In [ ]:
# R² на перемешанном target должен быть около 0 (STEP 9 ноутбука XGBoost, для любой модели)
evaluation.shuffled_target_check(Forecaster.from_recipe("xgboost"), split.train, split.test, val=split.val)

## 10. Сырые 15-минутные данные той же станции

Здесь ночные нули — настоящие, поэтому в спецификации `zeros: false`. Правило SVGP-ноутбука
«ноль = пропуск во всех колонках» удалило бы здесь все ночи (~51 % строк).

In [ ]:
RAW_XLSX = Path("../../1_FIGURES_PROJ/Data/Solar station site 1 (Nominal capacity-50MW).xlsx")
raw = load_station(RAW_XLSX, SPECS / "china_site1_raw.yaml")
raw_plan = Cleaner.from_spec(raw.spec).plan(raw)
print(raw, "| строк будет удалено:", raw_plan["rows_to_drop"])
analysis.find_blocks(raw, min_length=25, columns=["AIR_P", "RH"], nan=True, zeros=True)